# 项目 — 航空公司 AI 助手

现在我们把所学内容整合起来，为一家航空公司打造 AI 客户支持助手

In [ ]:
# 导入

# 导入标准库 os（操作系统相关，用来读环境变量 Environment Variables）
import os
# 导入 json：把 JSON（JavaScript Object Notation，一种常见数据格式）字符串和 Python 字典互转
import json
# 从 dotenv 导入 load_dotenv：把 .env 文件里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 openai 导入 OpenAI 客户端类：用它调用 Chat Completions 等 API（Application Programming Interface）
from openai import OpenAI
# 导入 Gradio：快速搭建可交互的 Web 演示界面（聊天框、按钮等）
import gradio as gr
# 导入 sqlite3：Python 自带的轻量数据库接口，用来读写本地 SQLite 文件
import sqlite3

In [ ]:
# 初始化

# 加载 .env 文件：把 API Key 等密钥读入进程环境（override=True 表示覆盖已有同名变量）
load_dotenv(override=True)

# 用 os.getenv 读取环境变量里的密钥；找不到时返回 None
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
# 选定本次实验使用的模型名称（model id）
MODEL = "gpt-4.1-mini"
# 创建 OpenAI 客户端；不传参时默认读环境变量里的 API Key
openai = OpenAI()

# 指定 SQLite 数据库文件路径（本地一个 .db 文件即可）
DB = "prices.db"

In [ ]:
# 系统消息（system message）：聊天场景下的角色设定，等价于 system prompt
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [ ]:
# 本地工具函数：按城市查询机票价格（给模型「动手」用）
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    # 连接数据库；with 结束后自动关闭连接，避免忘记关
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [ ]:
get_ticket_price("Paris")

In [ ]:
# 定义「工具」（tool / function calling）的 JSON Schema：告诉模型可以调用哪些函数、参数长什么样
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}
# 把工具定义放进 tools 列表，稍后传给 chat.completions.create，让模型决定是否调用
tools = [{"type": "function", "function": price_function}]
tools

In [ ]:

# Gradio 会调用的聊天回调：接收用户消息与历史，返回助手回复
def chat(message, history):
    # 把 Gradio 传来的聊天历史整理成 OpenAI messages 格式
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    # 组装 messages 列表：Chat Completions API 要求的对话格式（system / user / assistant）
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    # 调用 chat.completions.create：向大模型发一次对话请求并拿回复（同步、等全部生成完）
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

# 启动 Gradio ChatInterface：把你的 chat 函数挂到网页聊天窗口上
gr.ChatInterface(fn=chat, type="messages").launch()

In [ ]:
# Gradio 会调用的聊天回调：接收用户消息与历史，返回助手回复
def chat(message, history):
    # 把 Gradio 传来的聊天历史整理成 OpenAI messages 格式
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    # 组装 messages 列表：Chat Completions API 要求的对话格式（system / user / assistant）
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    # 调用 chat.completions.create：向大模型发一次对话请求并拿回复（同步、等全部生成完）
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    # 若 finish_reason 为 tool_calls，说明模型要求先调用本地工具再继续回答
    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        # 调用 chat.completions.create：向大模型发一次对话请求并拿回复（同步、等全部生成完）
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [ ]:
# 处理模型返回的 tool_calls：真正执行本地函数，再把结果回传给模型
def handle_tool_calls(message):
    responses = []
    # 模型可能一次请求多个工具；逐个解析参数并执行
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            # json.loads：把 JSON 字符串解析成 Python 字典/列表
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [ ]:
# 启动 Gradio ChatInterface：把你的 chat 函数挂到网页聊天窗口上
gr.ChatInterface(fn=chat, type="messages").launch()

## 再多了解一点 Gradio 实际做了什么：

1. Gradio 根据我们对 UI 的 Python 描述，构建一个前端 Svelte 应用
2. Gradio 启动一个基于 Starlette Web 框架的服务器，监听空闲端口，并提供该 Svelte 应用
3. Gradio 为我们的回调（如 chat()）创建后端路由，这些路由会调用我们的函数

当然，当 Gradio 生成前端应用时，它会确保 Submit 按钮调用正确的后端路由。

就是这样！

很简单，但结果却有种魔力。

# 让我们走向多模态！！

我们可以用 GPT-5 背后的图像生成模型 gpt-image-1-mini 来生成一些图片

把它放进一个叫 artist 的函数里。

### 价格提醒：我每生成一张图大约花费 3 美分——别疯狂生成图片！

In [ ]:
# 一些用于处理图像的导入

# 导入 base64：把二进制数据编码成可传输的文本（Base64）
import base64
# 从 io 导入 BytesIO：把字节数据当成「内存里的文件」来读写
from io import BytesIO
# 从 PIL（Pillow）导入 Image：处理图像对象
from PIL import Image

In [ ]:
# 调用图像生成 API，根据文字描述画出图片
def artist(city):
    # 调用图像生成接口（Images API），根据文字提示画图
    image_response = openai.images.generate(
            model="gpt-image-1-mini",
            # 拼出最终 prompt（提示词），送给模型
            prompt=f"An image representing a vacation in {city}, showing tourist spots and everything unique about {city}, in a vibrant pop-art style",
            size="1024x1024",
            n=1,
        )
    image_base64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))

In [ ]:
# 生成一张城市主题图并拿到 Image 对象
image = artist("New York City")
display(image)

In [ ]:
# 调用语音合成（TTS, Text-To-Speech），把文字转成语音
def talker(message):
    # 调用语音合成接口（TTS）：文字 → 音频
    response = openai.audio.speech.create(
      model="gpt-4o-mini-tts",
      voice="onyx",    # Also, try replacing onyx with alloy or coral
      input=message
    )
    return response.content

## 把这一切收个尾：

1. 具备图像与音频生成能力的多模态 AI 助手
2. 带数据库查询的工具调用
3. 迈向智能体（Agentic）工作流的一步


In [ ]:
# Gradio 会调用的聊天回调：接收用户消息与历史，返回助手回复
def chat(history):
    # 把 Gradio 传来的聊天历史整理成 OpenAI messages 格式
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    # 组装 messages 列表：Chat Completions API 要求的对话格式（system / user / assistant）
    messages = [{"role": "system", "content": system_message}] + history
    # 调用 chat.completions.create：向大模型发一次对话请求并拿回复（同步、等全部生成完）
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    cities = []
    image = None

    # 若 finish_reason 为 tool_calls，说明模型要求先调用本地工具再继续回答
    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses, cities = handle_tool_calls_and_return_cities(message)
        messages.append(message)
        messages.extend(responses)
        # 调用 chat.completions.create：向大模型发一次对话请求并拿回复（同步、等全部生成完）
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    # 从响应里取出第一条候选的 message.content（模型生成的文本）
    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}]

    voice = talker(reply)

    if cities:
        # 生成一张城市主题图并拿到 Image 对象
        image = artist(cities[0])
    
    return history, voice, image


In [ ]:
# 处理模型返回的 tool_calls：真正执行本地函数，再把结果回传给模型
def handle_tool_calls_and_return_cities(message):
    responses = []
    cities = []
    # 模型可能一次请求多个工具；逐个解析参数并执行
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            # json.loads：把 JSON 字符串解析成 Python 字典/列表
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            cities.append(city)
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses, cities

## Gradio UI 的 3 种类型

`gr.Interface` 用于标准、简单的 UI

`gr.ChatInterface` 用于标准聊天机器人 UI

`gr.Blocks` 用于自定义 UI，由你控制组件和回调

In [ ]:
# 回调（以及上面的 chat() 函数）

def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

# UI 定义

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        image_output = gr.Image(height=500, interactive=False)
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

# 将事件挂钩到回调

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, audio_output, image_output]
    )

# launch：启动本地 Web 服务并打开演示页面
ui.launch(inbrowser=True, auth=("ed", "bananas"))

# 练习与商业应用

再添加更多工具——或许可以模拟真正预订航班。已有同学完成，并在 community contributions 文件夹中提供了示例。

下一步：把这套做法应用到你的业务中。做一个带工具的多模态 AI 助手，能在工作中完成某项活动。客户支持助手？新员工入职助手？可能性太多了！另外，请查看单独 Notebook 中的 week2 周末练习。

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">我有一个特别请求</h2>
            <span style="color:#090;">
                我的编辑告诉我，学员在 Udemy 上给这门课评分会带来巨大影响——这是 Udemy 决定是否向其他人展示课程的主要方式之一。如果你能花一分钟评分，我将无比感激！无论如何——如果任何时候需要帮助，请随时通过 ed@edwarddonner.com 联系我。
            </span>
        </td>
    </tr>
</table>